In [1]:
import re, json
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

QUESTIONS_PDF = "Questions.pdf"
EASY_PDF = "Test PDF (Easy).pdf"
HARD_PDF = "Test PDF (Hard).pdf"

c:\Users\MLDL_02\anaconda3\envs\cenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_pdf(path):
    return "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)

raw_q    = read_pdf(QUESTIONS_PDF)
raw_easy = read_pdf(EASY_PDF)
raw_hard = read_pdf(HARD_PDF)
print(raw_hard[:400])   # check the noise in the hard file

Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%


In [3]:
# The hard PDF has junk like (_$$$%%%&&&())))( inserted mid-sentence, often wrapped over lines
NOISE = re.compile(r"(?:[_$%&()]\s*){5,}")

def clean_text(t):
    t = re.sub(r"\s+", " ", t)                    # join wrapped lines
    t = NOISE.sub(" ", t)                         # remove symbol noise
    t = re.sub(r"\s+\.\s+(?=[a-z])", " ", t)     # stray '.' left behind
    t = re.sub(r"\.(?=[A-Z])", ". ", t)           # ".Bangladesh" -> ". Bangladesh"
    t = re.sub(r"\s+([.,])", r"\1", t)            # space before punctuation
    return re.sub(r"\s+", " ", t).strip()

def parse_questions(raw):
    # also splits Q6, which contains two questions in one line
    raw = re.sub(r"\s+", " ", raw).replace("Questions:", "")
    qs = []
    for item in re.split(r"\s*\d+\.\s+", raw):
        qs += [q.strip() for q in re.split(r"(?<=\?)\s+", item) if q.strip().endswith("?")]
    return qs

questions    = parse_questions(raw_q)
easy_context = clean_text(raw_easy)
hard_context = clean_text(raw_hard)

print(len(questions), "questions")
print(hard_context[:500])

7 questions
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the. Padma River, which is often referred to as the lifeline of th


In [4]:
def make_chunks(text, size=3):
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [" ".join(sents[i:i+size]) for i in range(max(1, len(sents)-size+1))]

def top_context(question, chunks, vec, mat, k=2):
    sims = cosine_similarity(vec.transform([question]), mat)[0]
    idx = sorted(sims.argsort()[::-1][:k])
    return " ".join(chunks[i] for i in idx)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

MODEL = "deepset/roberta-base-squad2"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL).eval()

def qa(question, context, max_ans_len=30):
    enc = tok(question, context, return_tensors="pt",
              truncation="only_second", max_length=512, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    seq = enc.sequence_ids(0)
    with torch.no_grad():
        out = model(**enc)

    # only allow answer spans inside the context
    ctx_mask = torch.tensor([s == 1 for s in seq])
    start = out.start_logits[0].masked_fill(~ctx_mask, -1e9)
    end = out.end_logits[0].masked_fill(~ctx_mask, -1e9)

    # score every (start, end) pair; keep start <= end and a max answer length
    scores = torch.softmax(start, 0)[:, None] * torch.softmax(end, 0)[None, :]
    scores = torch.tril(torch.triu(scores), diagonal=max_ans_len - 1)

    i, j = divmod(int(scores.argmax()), scores.shape[1])
    return {"answer": context[offsets[i][0]:offsets[j][1]],
            "score": float(scores[i, j])}

c:\Users\MLDL_02\anaconda3\envs\cenv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
def answer_all(questions, context, k=2):
    chunks = make_chunks(context)
    vec = TfidfVectorizer(stop_words="english").fit(chunks)
    mat = vec.transform(chunks)
    results = []
    for q in questions:
        ctx = top_context(q, chunks, vec, mat, k)
        r = qa(question=q, context=ctx)
        results.append({"question": q, "answer": r["answer"].strip(" ."), "score": round(r["score"], 3)})
    return results

In [7]:
easy_results = answer_all(questions, easy_context)
for i, r in enumerate(easy_results, 1):
    print(f"{i}. {r['question']}\n   -> {r['answer']} ({r['score']})")
json.dump(easy_results, open("easy_answers.json", "w"), indent=2, ensure_ascii=False)

1. What is the capital city of Bangladesh?
   -> Dhaka (0.749)
2. Which river is considered the lifeline of Bangladesh?
   -> Padma River (0.373)
3. In which year did Bangladesh gain independence?
   -> 1971 (0.579)
4. What is the official language of Bangladesh?
   -> Bengali (0.967)
5. What is the name of the world’s largest mangrove forest located in Bangladesh?
   -> Sundarbans (0.456)
6. Which currency is used in Bangladesh?
   -> Bangladeshi Taka (0.327)
7. What is the national animal of Bangladesh?
   -> Royal Bengal Tiger (0.556)


In [8]:
hard_results = answer_all(questions, hard_context)
for i, r in enumerate(hard_results, 1):
    print(f"{i}. {r['question']}\n   -> {r['answer']} ({r['score']})")
json.dump(hard_results, open("hard_answers.json", "w"), indent=2, ensure_ascii=False)

1. What is the capital city of Bangladesh?
   -> Dhaka (0.749)
2. Which river is considered the lifeline of Bangladesh?
   -> Padma River (0.664)
3. In which year did Bangladesh gain independence?
   -> 1971 (0.579)
4. What is the official language of Bangladesh?
   -> Bengali (0.968)
5. What is the name of the world’s largest mangrove forest located in Bangladesh?
   -> Sundarbans (0.44)
6. Which currency is used in Bangladesh?
   -> Bangladeshi Taka (0.568)
7. What is the national animal of Bangladesh?
   -> Royal Bengal Tiger (0.556)


In [9]:
for e, h in zip(easy_results, hard_results):
    print(f"{'OK ' if e['answer']==h['answer'] else 'DIFF'} | {e['answer']}  |  {h['answer']}")

OK  | Dhaka  |  Dhaka
OK  | Padma River  |  Padma River
OK  | 1971  |  1971
OK  | Bengali  |  Bengali
OK  | Sundarbans  |  Sundarbans
OK  | Bangladeshi Taka  |  Bangladeshi Taka
OK  | Royal Bengal Tiger  |  Royal Bengal Tiger
